In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

**STRUCTURED TOOL**

In [3]:
class TenX_Schema(BaseModel):
    a : float = Field(required=True, description='An number to multiply')

In [4]:
def TenX_func(a):
    return a*10

In [5]:
TenX_tool = StructuredTool.from_function(
    func=TenX_func,
    args_schema=TenX_Schema,
    name='TenX',
    description='Multiplies the input number with 10'
)

In [8]:
TenX_tool.invoke({'a':1.00000009})

10.0000009

**CREATING TOOL USING BASE TOOL** **ADVANTANGE ASYNC FUNC**

In [19]:
from langchain_core.tools import BaseTool
from typing import Type

In [36]:
class Base_TenX_Schema(BaseModel):
    a : float = Field(required=True, description='An number to multiply 10 times')

In [37]:
class Base_Tool_func(BaseTool):
    name : str = 'Multiply'
    description : str = 'multiplies the number with 10'

    args_schema : Type[BaseModel] = Base_TenX_Schema
    def _run(self, a:float) -> float:
        return 10 * a

In [38]:
TenX_tool_basetool = Base_Tool_func()

In [39]:
TenX_tool_basetool.invoke({'a':53})

530.0

In [40]:
TenX_tool_basetool.args

{'a': {'description': 'An number to multiply 10 times',
  'title': 'A',
  'type': 'number'}}

**LLM Conn**

In [27]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os
load_dotenv()

True

In [41]:
llm = HuggingFaceEndpoint(repo_id=os.getenv('hf_model'))
hf_model = ChatHuggingFace(llm=llm)

In [42]:
hf_model_with_tools = hf_model.bind_tools([TenX_tool_basetool])

In [35]:
hf_model_with_tools.invoke('if you had to describe Fang Yuan from Reverend Insanity, what would it be?')

AIMessage(content='Fang Yuan from "Reverend Insanity" is a complex and multifaceted character. He is the main protagonist and a white reverend who serves as the head of a private orphanage in a small town. Fang Yuan is initially portrayed as a seemingly earnest and kind-hearted figure who dedicates his life to caring for the orphans under his care. However, the story delves into his darker past, revealing a disturbing ritual known as the "Seven Copies of Fate," which he uses to sacrifice children to an unknown entity in exchange for power and immortality. This dark aspect of his character makes him a tragic antihero, grappling with his moral decay while attempting to maintain his role as a guardian of the children. His journey is a psychological and ethical exploration of the lengths to which one might go for personal gain and immortality.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 217, 'total_tokens': 387}, 'model_name': 'Qwen

if the tool is useful, the llm returns the suggestion of the tool that it has access to
(we bind the tools to llm so that it can see the description)

In [56]:
hf_model_with_tools.invoke('10 x 4.6')

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":4.6}', 'name': 'Multiply', 'description': None}, 'id': 'call_7sbj781v5534c2nkds24q0ez', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 210, 'total_tokens': 231}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ffbbe-7bda-7890-9fdd-e3937a64735b-0', tool_calls=[{'name': 'Multiply', 'args': {'a': 4.6}, 'id': 'call_7sbj781v5534c2nkds24q0ez', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 210, 'output_tokens': 21, 'total_tokens': 231})

In [52]:
llm_resultofmul = hf_model_with_tools.invoke('10 x 4.6')

In [58]:
llm_resultofmul.tool_calls

[{'name': 'Multiply',
  'args': {'a': 4.6},
  'id': 'call_o64tvmhfs328r6opjx18416n',
  'type': 'tool_call'}]

In [53]:
llm_mul_tool_call = llm_resultofmul.tool_calls[0]

In [59]:
toolcall_execution_message_returned = TenX_tool_basetool.invoke(llm_mul_tool_call)

In [60]:
toolcall_execution_message_returned

ToolMessage(content='46.0', name='Multiply', tool_call_id='call_o64tvmhfs328r6opjx18416n')

In [61]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage